In [0]:
catalog = "CineData_Analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# SILVER - TB_INFO_FILMES

## Regras de Negócio

A tabela `tb_info_filmes` consolida as informações cadastrais dos filmes provenientes da camada Bronze em uma estrutura padronizada para consumo analítico.

### Qualidade e Unicidade dos Dados

- Registros sem identificador de filme são descartados, pois não permitem garantir a unicidade dos dados.
- Em caso de múltiplos registros para o mesmo filme, é mantida apenas a versão mais recente com base na coluna `ingestion_datetime`, assegurando a utilização dos dados mais atualizados.

### Tratamento e Padronização de Status

- A coluna de status passa por um processo de normalização para eliminar inconsistências na origem.
- São removidos espaços excedentes e caracteres indesejados.
- Os valores são padronizados para uma única convenção de escrita.
- Após a normalização, os status são traduzidos para português para facilitar a interpretação por usuários de negócio.

Mapeamento aplicado:

| Status Original | Status Final |
|---------------|--------------|
| Released | Lançado |
| Post Production | Pós-Produção |
| In Production | Em Produção |
| Planned | Planejado |
| Rumored | Rumores |
| Canceled | Cancelado |

Valores não reconhecidos são classificados como **"Não Informado"**.

### Tratamento da Data de Lançamento

- A coluna `release_date` pode possuir diferentes formatos de data provenientes das fontes de origem.
- Os valores são convertidos para o tipo `date`, garantindo consistência para análises temporais.
- São tratados os formatos identificados durante a análise exploratória dos dados.

### Coluna Derivada

- A coluna `ano_lancamento` é derivada da data de lançamento.
- Seu objetivo é facilitar análises históricas, agregações e segmentações por ano.

### Padronização de Nomenclatura

As colunas foram renomeadas para português conforme o modelo de dados da camada Silver:

| Origem | Destino |
|---------|---------|
| id | id_filme |
| title | titulo |
| original_title | titulo_original |
| release_date | data_lancamento |
| runtime | duracao_minutos |
| original_language | idioma_original |
| status | status_filme

In [0]:
from pyspark.sql import functions as F, Row
from pyspark.sql.window import Window

df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

window_filmes = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("ingestion_datetime").desc()
    )
)

tb_info_filmes = (
    df_bronze_info
    .filter(F.col("id").isNotNull())
    .withColumn("status_normalizado", F.upper( F.trim( F.regexp_replace( F.col("status"),"-"," "))))
    .withColumn("status", F.when( F.col("status_normalizado") == "RELEASED", "Lançado")
                .when( F.col("status_normalizado") == "POST PRODUCTION", "Pós-produção")
                .when( F.col("status_normalizado") == "IN PRODUCTION", "Em Produção")
                .when( F.col("status_normalizado") == "PLANNED", "Planejado")
                .when( F.col("status_normalizado") == "RUMORED", "Rumores")
                .when( F.col("status_normalizado") == "CANCELED", "Cancelado")
                .otherwise("Não Informado"))
    .drop("status_normalizado")
    .withColumn("row_num",F.row_number().over(window_filmes))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("data_lancamento", F.when( F.col("release_date").rlike("^\\d{4}"), F.to_date( F.col("release_date"), "yyyy-MM-dd"))
                .when( F.col("release_date").rlike("^\\d{2}-\\d{2}-\\d{4}$"), F.to_date( F.col("release_date"), "MM-dd-yyyy"))
                .when( F.col("release_date").rlike("^\\d{2}/\\d{2}/\\d{4}$"), F.to_date( F.col("release_date"), "dd/MM/yyyy"))
                .otherwise(None))
    .withColumn("ano_lancamento",F.year("data_lancamento"))
    .select("id", "title", "original_title", "data_lancamento", "ano_lancamento", "runtime", "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

tb_info_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_info_filmes"
    )

# SILVER - TB_FINANCEIRO_FILMES

## Regras de Negócio

A tabela `tb_financeiro_filmes` consolida as informações financeiras dos filmes provenientes da camada Bronze em uma estrutura padronizada para análises de rentabilidade, retorno financeiro e desempenho econômico.

### Tratamento de Valores Ausentes

- Valores textuais que representam ausência de informação são convertidos para `NULL`.
- Foram tratados valores como:
  - `Unknown`
  - `Não Informado`
  - `N/A`
- Essa padronização evita que informações inválidas sejam consideradas em cálculos financeiros.

### Higienização dos Valores Monetários

- As colunas financeiras passam por um processo de limpeza para remover elementos que impedem a conversão para tipo numérico.
- São removidos:
  - Símbolos monetários (`$`)
  - Identificadores de moeda (`USD`)
  - Espaços excedentes

### Padronização de Escalas Financeiras

Os valores financeiros podem ser recebidos em diferentes escalas de representação.

Foram aplicadas as seguintes conversões:

| Valor de Origem | Valor Convertido |
|----------------|------------------|
| 20.0M | 20.000.000 |
| 90.0M | 90.000.000 |
| 100.0K | 100.000 |
| 250.0K | 250.000 |

- Valores contendo o sufixo **M** são convertidos para milhões.
- Valores contendo o sufixo **K** são convertidos para milhares.

### Conversão de Tipo

- Após a higienização dos dados, os valores financeiros são convertidos para tipo numérico decimal.
- Essa conversão permite a realização de cálculos financeiros, agregações e análises estatísticas.

### Conversão Cambial

- Os valores originalmente armazenados em dólares americanos (USD) são convertidos para reais (BRL).
- A conversão utiliza a cotação de dólar mais recente disponível na tabela de câmbio da camada Bronze.

Foram geradas as colunas:

| Coluna | Descrição |
|----------|----------|
| orcamento_brl | Orçamento convertido para Reais |
| receita_brl | Receita convertida para Reais |

### Métricas Derivadas

Foram calculados indicadores financeiros para análise de desempenho econômico dos filmes.

#### Lucro em Dólar

Representa a diferença entre a receita e o orçamento em USD.

```text
lucro_usd = receita_usd - orcamento_usd
```

#### Lucro em Real

Representa a diferença entre a receita e o orçamento em BRL.

```text
lucro_brl = receita_brl - orcamento_brl
```

#### Margem de Lucro Percentual

Representa a participação percentual do lucro em relação à receita.

```text
margem_lucro_pct =
(lucro_usd / receita_usd) * 100
```

### Integridade dos Cálculos

- Valores ausentes permanecem nulos após as transformações.
- Os cálculos respeitam a existência de dados válidos.
- Operações potencialmente inválidas, como divisões por zero, são evitadas para preservar a consistência dos resultados.

### Padronização de Nomenclatura

As colunas foram renomeadas conforme o modelo de dados da camada Silver:

| Origem | Destino |
|---------|---------|
| id | id_filme |
| budget | orcamento_usd |
| revenue | receita_usd |
| orcamento_usd | orcamento_brl |
| receita_usd | receita_brl |

In [0]:
df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

tb_financeiro_filmes = (
    df_bronze_financials
    


    .withColumn("orcamento_usd",
    F.when(F.col("budget").isin("N/A","Unknown","Não Informado"),None)
    .when(F.col("budget").contains("M"),F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("budget"),"\\$",""),"USD",""),"M","").cast("double") * 1000000)
    .when(F.col("budget").contains("K"),F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("budget"),"\\$",""),"USD",""),"K","").cast("double") * 1000)
    .otherwise(F.trim(F.regexp_replace(F.regexp_replace(F.col("budget"),"\\$",""),"USD","")).cast("double")))


    .withColumn("receita_usd",F.when(F.col("revenue").isin("Unknown","Não Informado"),None)
    .otherwise(F.col("revenue").cast("double")))


    .withColumn("orcamento_brl",F.col("orcamento_usd") * cotacao_dolar)
    .withColumn("receita_brl",F.col("receita_usd") * cotacao_dolar)


    .withColumn("lucro_usd",F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl",F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn("margem_lucro_pct",F.when(F.col("receita_usd") > 0,(F.col("lucro_usd")/ F.col("receita_usd")) * 100))


    .withColumnRenamed("id", "id_filme")

    .select("id_filme","orcamento_usd","receita_usd","orcamento_brl","receita_brl","lucro_usd","lucro_brl","margem_lucro_pct","ingestion_datetime")
)

tb_financeiro_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_financeiro_filmes"
    )

# SILVER - TB_METRICAS_ENGAJAMENTO

## Regras de Negócio

A tabela `tb_metricas_engajamento` consolida indicadores de popularidade, avaliações e volume de votos provenientes das plataformas TMDB e IMDb, disponibilizando métricas confiáveis para análises de desempenho, engajamento e recepção dos filmes.

### Padronização de Nomenclatura

As colunas foram renomeadas para português conforme o modelo de dados da camada Silver.

| Origem | Destino |
|---------|---------|
| id | id_filme |
| popularity | popularidade |
| vote_average | nota_media_tmdb |
| vote_count | qtd_votos_tmdb |
| averageRating | nota_media_imdb |
| numVotes | qtd_votos_imdb |

### Tratamento da Popularidade

- A coluna de popularidade apresentou inconsistências nos separadores decimais.
- Valores contendo vírgula foram padronizados para ponto antes da conversão de tipo.
- Após a padronização, a coluna foi convertida para tipo numérico decimal.

Exemplos de tratamento:

| Valor Original | Valor Tratado |
|---------------|---------------|
| 154,34 | 154.34 |
| 54,522 | 54.522 |
| 72.735 | 72.735 |

### Correção de Inconsistências de Estrutura (Column Shift)

- Foram identificados registros contendo descrições textuais em colunas destinadas a métricas numéricas.
- Esses registros são resultado de deslocamentos de dados na origem.
- Valores incompatíveis com o formato esperado são tratados como nulos para preservar a integridade analítica da tabela.

### Conversão Segura de Tipos

As métricas foram convertidas para tipos numéricos adequados para análises estatísticas e agregações.

| Coluna | Tipo |
|---------|---------|
| popularidade | Double |
| nota_media_tmdb | Double |
| qtd_votos_tmdb | Integer |
| nota_media_imdb | Double |
| qtd_votos_imdb | Integer |

### Validação das Notas

As plataformas TMDB e IMDb utilizam escala de avaliação entre 0 e 10.

Foram aplicadas as seguintes regras:

- Valores menores que 0 são considerados inválidos.
- Valores maiores que 10 são considerados inválidos.
- Valores inválidos são convertidos para NULL.

### Validação das Métricas de Engajamento

- Popularidade não pode assumir valores negativos.
- Quantidade de votos não pode assumir valores negativos.
- Registros inválidos são tratados como NULL para evitar distorções nos indicadores analíticos.

### Qualidade dos Dados

- Valores incompatíveis com a tipagem esperada são tratados de forma segura.
- Registros com inconsistências não interrompem o processamento da pipeline.
- A coluna `ingestion_datetime` é mantida para garantir rastreabilidade, auditoria e controle da origem dos dados.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- id_filme
- popularidade
- nota_media_tmdb
- qtd_votos_tmdb
- nota_media_imdb
- qtd_votos_imdb
- ingestion_datetime

In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

tb_metricas_engajamento = (
    df_bronze_metrics

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
    .withColumn("popularidade",F.regexp_replace(F.col("popularidade"),",","."))
    .withColumn("popularidade",F.when(F.col("popularidade").rlike("^[0-9]+\\.?[0-9]*$"),F.col("popularidade").cast("double")))
    .withColumn("nota_media_tmdb",F.when(F.col("nota_media_tmdb").rlike("^[0-9]+\\.?[0-9]*$"),F.col("nota_media_tmdb").cast("double")))
    
    .withColumn("qtd_votos_tmdb",F.when(F.col("qtd_votos_tmdb").rlike("^[0-9]+$"),F.col("qtd_votos_tmdb").cast("int")))

    .withColumn("nota_media_imdb",F.when(F.col("nota_media_imdb").rlike("^[0-9]+\\.?[0-9]*$"),F.col("nota_media_imdb").cast("double")))

    .withColumn("qtd_votos_imdb",F.when(F.col("qtd_votos_imdb").rlike("^[0-9]+$"),F.col("qtd_votos_imdb").cast("int")))

    .withColumn("nota_media_tmdb",F.when((F.col("nota_media_tmdb") >= 0)& (F.col("nota_media_tmdb") <= 10),F.col("nota_media_tmdb")))
    
    .withColumn("nota_media_imdb",F.when((F.col("nota_media_imdb") >= 0)& (F.col("nota_media_imdb") <= 10),F.col("nota_media_imdb")))

    .withColumn("popularidade",F.when(F.col("popularidade") >= 0,F.col("popularidade")))

    .withColumn("qtd_votos_tmdb",F.when(F.col("qtd_votos_tmdb") >= 0,F.col("qtd_votos_tmdb")))

    .withColumn("qtd_votos_imdb",F.when(F.col("qtd_votos_imdb") >= 0,F.col("qtd_votos_imdb")))

)

tb_metricas_engajamento.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_metricas_engajamento"
    )

# SILVER - TB_AVALIACOES_USUARIOS

## Regras de Negócio

A tabela `tb_avaliacoes_usuarios` consolida as avaliações realizadas pelos usuários sobre os filmes, disponibilizando informações padronizadas para análises de satisfação, percepção do público e experiência dos usuários.

### Padronização de Nomenclatura

As colunas foram renomeadas para português conforme o modelo de dados da camada Silver.

| Origem | Destino |
|---------|---------|
| id | id_filme |
| nome | nome_usuario |
| nota | nota_usuario |
| comentario | comentario_usuario |

### Tratamento de Duplicidades

- Registros integralmente duplicados são removidos.
- A unicidade da avaliação é considerada com base na combinação dos atributos:
  - id_filme
  - nome_usuario
  - nota_usuario
  - comentario_usuario

### Validação das Avaliações

- As notas dos usuários devem respeitar a escala de avaliação entre 0 e 10.
- Valores inferiores a 0 são considerados inválidos.
- Valores superiores a 10 são considerados inválidos.
- Notas fora da faixa permitida são convertidas para `NULL` para preservar a qualidade analítica dos dados.

### Padronização dos Comentários

- Comentários nulos são substituídos pelo texto padronizado **"Sem comentário"**.
- Comentários compostos apenas por espaços em branco também são substituídos por **"Sem comentário"**.
- A padronização garante consistência na análise textual das avaliações dos usuários.

Exemplos de tratamento:

| Valor Original | Valor Tratado |
|---------------|---------------|
| NULL | Sem comentário |
| "" | Sem comentário |
| "     " | Sem comentário |

### Qualidade dos Dados

- Registros inconsistentes são tratados sem interromper o processamento da pipeline.
- A coluna `ingestion_datetime` é mantida para garantir rastreabilidade e auditoria dos dados.
- As transformações garantem uma estrutura adequada para análises de satisfação e comportamento dos usuários.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- id_filme
- nome_usuario
- nota_usuario
- comentario_usuario
- ingestion_datetime

In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

tb_avaliacoes_usuarios = (
    df_bronze_reviews

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")

    .dropDuplicates(["id_filme","nome_usuario","nota_usuario","comentario_usuario"])

    .withColumn("nota_usuario",F.when((F.col("nota_usuario") >= 0)& (F.col("nota_usuario") <= 10),F.col("nota_usuario")))

    .withColumn("comentario_usuario",F.when(F.col("comentario_usuario").isNull()  | (F.trim(F.col("comentario_usuario")) == ""),"Sem comentário")
    .otherwise(F.col("comentario_usuario")))

    .dropDuplicates(["id_filme","nome_usuario","nota_usuario","comentario_usuario"])

)

tb_avaliacoes_usuarios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_avaliacoes_usuarios"
    )

# SILVER - TB_GENEROS

## Regras de Negócio

A tabela `tb_generos` consolida os gêneros associados aos filmes, transformando uma estrutura multivalorada da origem em uma estrutura normalizada, adequada para análises temáticas, segmentações e classificações cinematográficas.

### Padronização dos Separadores

- A coluna `genres` apresenta inconsistências na forma de armazenamento dos gêneros.
- Foram identificados diferentes separadores para representar múltiplos gêneros no mesmo registro.
- Os separadores foram padronizados antes do processamento dos dados para garantir consistência na etapa de desmembramento.

Exemplos de tratamento:

| Valor Original | Valor Padronizado |
|---------------|-------------------|
| Action\|Adventure\|Fantasy | Action,Adventure,Fantasy |
| Action;Comedy;Adventure | Action,Comedy,Adventure |

### Normalização dos Gêneros

- A origem armazena múltiplos gêneros em uma única coluna.
- Foi aplicado o processo de separação dos gêneros utilizando operações de `split`.
- Em seguida, foi aplicada a operação de `explode`, transformando cada gênero em um registro independente.

Exemplo:

| Origem |
|---------|
| Action, Adventure, Comedy |

Transformado em:

| id_filme | genero |
|----------|---------|
| 293660 | Action |
| 293660 | Adventure |
| 293660 | Comedy |

### Tratamento de Espaços em Branco

- Após a separação dos gêneros, foram removidos espaços excedentes no início e no final dos valores.
- O tratamento evita duplicidades causadas por diferenças de formatação.

Exemplo:

| Valor Original | Valor Tratado |
|---------------|---------------|
| " Adventure" | Adventure |
| "Comedy " | Comedy |

### Correção de Inconsistências Estruturais

Foram identificados problemas de qualidade decorrentes de falhas de estrutura na origem (**Column Shift**).

Entre os valores indevidos encontrados estavam:

- Valores numéricos deslocados
- Caminhos de arquivos de imagem (`.jpg`)
- Idiomas
- Nomes de pessoas
- Empresas
- Trechos descritivos de filmes
- Conteúdos textuais que não representam gêneros cinematográficos

Esses registros foram removidos por não pertencerem ao domínio de gêneros.

### Validação de Domínio

Foram mantidos apenas valores compatíveis com gêneros cinematográficos válidos.

Exemplos de gêneros preservados:

- Action
- Adventure
- Animation
- Comedy
- Crime
- Documentary
- Drama
- Family
- Fantasy
- History
- Horror
- Music
- Mystery
- Romance
- Science Fiction
- TV Movie
- Thriller
- War
- Western

### Qualidade dos Dados

- Registros incompatíveis com o domínio de gêneros foram descartados.
- Cada registro da tabela final representa exatamente um gênero associado a um filme.
- A coluna `ingestion_datetime` foi mantida para rastreabilidade e auditoria dos dados.
- O processamento garante consistência e confiabilidade para análises temáticas na camada Gold.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- id_filme
- genero
- ingestion_datetime

In [0]:
df_bronze_credits_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

tb_generos = (
    df_bronze_credits_tags

    .select("id","genres","ingestion_datetime")

    .withColumn("genres",F.regexp_replace(F.regexp_replace(F.col("genres"),"\\|",","),";",","))

    .withColumn("genero",F.explode(F.split(F.col("genres"),",")))

    .withColumn("genero",F.trim(F.col("genero")))

    .filter(F.col("genero").isin("Action","Adventure","Animation","Comedy","Crime","Documentary","Drama","Family","Fantasy","History","Horror","Music","Mystery","Romance","Science Fiction","TV Movie","Thriller","War","Western"))
    .withColumnRenamed("id", "id_filme")
    .select("id_filme","genero","ingestion_datetime")

)

tb_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_generos"
    )

# SILVER - TB_PESSOAS_EMPRESAS

## Regras de Negócio

A tabela `tb_pessoas_empresas` consolida informações de atores, diretores, roteiristas e produtoras associadas aos filmes, unificando diferentes tipos de entidades em uma única dimensão padronizada para análises relacionais e cruzamentos de informações.

### Consolidação das Entidades

As informações provenientes das colunas:

- cast
- directors
- writers
- production_companies

foram consolidadas em uma única tabela.

Cada registro representa uma única entidade associada a um filme.

### Mapeamento dos Tipos de Entidade

As entidades foram categorizadas conforme sua função ou participação no filme.

| Coluna de Origem | Tipo de Entidade |
|------------------|------------------|
| cast | Ator |
| directors | Diretor |
| writers | Roteirista |
| production_companies | Produtora |

### Normalização dos Registros

As colunas de origem armazenam múltiplos valores em um único campo.

Foi aplicado o processo de:

- Separação dos valores utilizando `split`
- Expansão dos registros utilizando `explode`

Esse processo garante que cada linha da tabela represente exatamente uma entidade relacionada a um filme.

Exemplo:

| Origem |
|---------|
| Ryan Reynolds, Morena Baccarin, Ed Skrein |

Transformado em:

| id_filme | entidade | tipo_entidade |
|-----------|-----------|----------------|
| 293660 | Ryan Reynolds | Ator |
| 293660 | Morena Baccarin | Ator |
| 293660 | Ed Skrein | Ator |

### Padronização dos Valores

- Espaços excedentes foram removidos dos nomes das entidades após o processo de separação.
- Os valores foram mantidos com a capitalização original quando já apresentavam formatação consistente.
- O tratamento evita inconsistências causadas por diferenças de preenchimento nos registros de origem.

### Remoção de Duplicidades

- Foram identificados registros duplicados após a consolidação das entidades.
- A deduplicação foi realizada utilizando a combinação:

  - id_filme
  - entidade
  - tipo_entidade

- O processo garante que cada relação entre filme e entidade seja representada apenas uma única vez.

### Modelo Unificado

As informações de elenco, direção, roteiro e produtoras foram integradas em uma única estrutura dimensional, categorizada pelo campo `tipo_entidade`.

Essa abordagem simplifica consultas analíticas e reduz a complexidade de cruzamento entre diferentes estruturas da origem.

### Qualidade dos Dados

- Registros duplicados foram eliminados.
- Espaços excedentes foram removidos.
- A estrutura resultante garante uma única representação para cada entidade vinculada ao filme.
- A coluna `ingestion_datetime` foi mantida para fins de rastreabilidade e auditoria dos dados.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- id_filme
- entidade
- tipo_entidade
- ingestion_datetime

In [0]:
tb_atores = (
    df_bronze_credits_tags

    .select("id","cast","ingestion_datetime")

    .withColumn("entidade",F.explode(F.split(F.col("cast"),",")))

    .withColumn("entidade",F.trim(F.col("entidade")))

    .withColumn("tipo_entidade",F.lit("Ator"))

    .select("id","entidade","tipo_entidade","ingestion_datetime")
)

tb_diretores = (
    df_bronze_credits_tags

    .select("id","directors","ingestion_datetime")

    .withColumn("entidade",F.explode(F.split(F.col("directors"),",")))

    .withColumn("entidade",F.trim(F.col("entidade")))

    .withColumn("tipo_entidade",F.lit("Diretor"))

    .select("id","entidade","tipo_entidade","ingestion_datetime"
)
)

tb_roteiristas = (
    df_bronze_credits_tags

    .select("id","writers","ingestion_datetime")

    .withColumn("entidade",F.explode(F.split(F.col("writers"),",")))

    .withColumn("entidade",F.trim(F.col("entidade")))

    .withColumn("tipo_entidade",F.lit("Roteirista"))

    .select("id","entidade","tipo_entidade","ingestion_datetime")
)

tb_produtoras = (
    df_bronze_credits_tags

    .select("id","production_companies","ingestion_datetime")

    .withColumn("entidade",F.explode(F.split(F.col("production_companies"),",")))

    .withColumn("entidade",F.trim(F.col("entidade")))

    .withColumn("tipo_entidade",F.lit("Produtora"))

    .select("id","entidade","tipo_entidade","ingestion_datetime")
)

tb_pessoas_empresas = (
    tb_atores
    .unionByName(tb_diretores)
    .unionByName(tb_roteiristas)
    .unionByName(tb_produtoras)
    .dropDuplicates(["id","entidade","tipo_entidade"])

    .withColumnRenamed("id","id_filme")

    .select("id_filme","entidade","tipo_entidade","ingestion_datetime")
)

tb_pessoas_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_pessoas_empresas"
    )


# SILVER - TB_COTACAO_DOLAR

## Regras de Negócio

A tabela `tb_cotacao_dolar` consolida o histórico da cotação do dólar em uma série temporal contínua, disponibilizando informações padronizadas para utilização em cálculos financeiros e conversão monetária nas demais tabelas analíticas do projeto.

### Padronização da Data

- A coluna `data_hora_cotacao` foi convertida para a coluna `data_cotacao`.
- O objetivo é representar a cotação em nível diário, removendo a granularidade de horário presente na origem.

Exemplo:

| Origem | Resultado |
|---------|---------|
| 2026-10-09 13:07:30 | 2026-10-09 |

### Tratamento de Duplicidades

- Foram identificados múltiplos registros para uma mesma data de cotação.
- Os registros foram deduplicados de forma a manter apenas uma cotação por dia.

Esse tratamento garante consistência na construção da série temporal.

### Construção da Série Temporal Contínua

- A API do Banco Central não disponibiliza cotações em finais de semana e feriados.
- Para evitar lacunas temporais, foi gerado um calendário contínuo contendo todas as datas compreendidas entre a menor e a maior data disponível na origem.
- Esse processo garante uma sequência diária sem interrupções.

Exemplo:

#### Série original

| Data |
|---------|
| 2026-10-09 |
| 2026-10-13 |
| 2026-10-14 |
| 2026-10-15 |

#### Série contínua

| Data |
|---------|
| 2026-10-09 |
| 2026-10-10 |
| 2026-10-11 |
| 2026-10-12 |
| 2026-10-13 |
| 2026-10-14 |
| 2026-10-15 |

### Preenchimento de Datas Ausentes (Forward Fill)

- As datas sem cotação foram preenchidas utilizando a última cotação válida disponível.
- Foi aplicada a técnica de **Forward Fill**, garantindo continuidade da série histórica mesmo nos períodos sem negociação.

Exemplo:

| Data | Cotação |
|---------|---------|
| 2026-10-09 | 4.9886 |
| 2026-10-10 | 4.9886 |
| 2026-10-11 | 4.9886 |
| 2026-10-12 | 4.9886 |
| 2026-10-13 | 5.0090 |

### Garantia de Integridade da Série

- Após a aplicação do Forward Fill, todas as datas do intervalo passam a possuir cotação válida.
- A série resultante não apresenta lacunas temporais.
- Não permanecem registros com valores nulos na coluna de cotação.

### Qualidade dos Dados

- Existe apenas uma cotação por dia.
- Todas as datas do período analisado possuem valor preenchido.
- A coluna `ingestion_datetime` foi preservada para fins de rastreabilidade e auditoria dos dados.
- A tabela passa a servir como referência única para cálculos de conversão cambial realizados na camada Silver e consumidos pela camada Gold.

### Estrutura Final da Tabela

A tabela é composta pelas seguintes colunas:

- data_cotacao
- cotacao_compra
- ingestion_datetime

In [0]:
tb_cotacao_dolar = (
    df_bronze_cotacao

    .withColumn("data_cotacao",F.to_date(F.col("data_hora_cotacao")))

    .select("data_cotacao","cotacao_compra","ingestion_datetime")

    .dropDuplicates(["data_cotacao"])

    
)

calendario = spark.sql("""
SELECT explode(
    sequence(
        to_date('2026-10-09'),
        to_date('2026-10-15'),
        interval 1 day
    )
) AS data_cotacao
""")

tb_cotacao_dolar = (
    calendario
    .join(tb_cotacao_dolar,on="data_cotacao",how="left")
)

from pyspark.sql.window import Window

window_cotacao = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, 0)
)

tb_cotacao_dolar = (
    tb_cotacao_dolar

    .withColumn("cotacao_compra",F.last("cotacao_compra",ignorenulls=True).over(window_cotacao))
)

tb_cotacao_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_cotacao_dolar"
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
